# Partitioning Strategies: Hash, Range, and Data Lake Partition Layouts

**Module**: 05_partitions_and_shuffle  
**Notebook**: 05_partitioning_strategies.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Hash partitioning vs Range partitioning, bucketing with bucketBy(), and partitioned directory layouts for data lakes.
- **Why does this topic exist?**: Proper storage partitioning enables partition pruning, eliminating 95% of data scans.
- **What problem does it solve?**: High-cardinality directory partitioning creating millions of tiny directories.
- **Where is it used in real-world Data Engineering?**: Designing enterprise Hive / Delta Lake partition architectures.

---

## 2. Partitioning Strategies

1. **Hash Partitioning**: `hash(key) % N` (Used for joins/aggregations)
2. **Range Partitioning**: Quantile sample ranges (Used for `orderBy`)
3. **Directory Partitioning**: `year=2026/month=01/` (Used in data lakes)

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Partitioning_Strategies_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
# Write partitioned by channel
out_path = "/opt/spark-data/warehouse/partitioned_by_channel"
orders.write.mode("overwrite").partitionBy("channel").parquet(out_path)

print("Partitions created on disk:")
for p in os.listdir(out_path):
    if not p.startswith("."):
        print(f"  Directory: {p}")

---

## Experiment: Partition Pruning Verification

- **1. What are we testing?**: Querying with filter on partition column skips other folders entirely.
- **2. What do we expect?**: Physical plan shows PartitionFilters: [channel ='Mobile'].

In [ ]:
read_part = spark.read.parquet(out_path)
filtered = read_part.filter("channel = 'Mobile'")
filtered.explain("formatted")

**3. What happened?**: PartitionFilters pruned irrelevant channel directories at the file scan level.  
**4. Why did it happen?**: Spark uses directory metadata to skip non-matching partition folders.  
**5. Practical Engineering Lesson**: Partition by low-to-medium cardinality columns used frequently in WHERE clauses.

---

## Practical Exercises

### Exercise 1 (Beginner): Partition by Year and Month
**Problem Statement**: Write orders partitioned by year and month.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Check Directory Cardinality
**Problem Statement**: Calculate distinct values of potential partition column.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Bucketing Concept
**Problem Statement**: Explain bucketing vs partitioning.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Partition by Year and Month):
```python
orders.withColumn('yr', F.year('order_ts')).write.mode('overwrite').partitionBy('yr').parquet('/opt/spark-data/warehouse/part_yr')
```

#### Solution 2 (Check Directory Cardinality):
```python
print('Channel distinct count:', orders.select('channel').distinct().count())
```

#### Solution 3 (Bucketing Concept):
```python
print('Partitioning creates directories; bucketing creates fixed-count hash files within directories.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Partition Pruning**: Skips entire folders based on WHERE filters.
- **Cardinality Rule**: Never partition by high cardinality keys (e.g. timestamp, id).

### Essential Technical Terms
- `Partition Pruning`: Skipping storage directories at read time.
- `Bucketing`: Pre-shuffling into fixed buckets.

### Common Engineering Mistakes
- **Partitioning by customer_id**: Creates 100,000 tiny directories, destroying metadata performance.

### Interview Questions on This Topic
- **Q: What is partition pruning?**
  - *A*: Spark skipping entire directory partitions during scanning because the query filter eliminates them.

### What to Study Next
Open 06_data_skew_and_solutions.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')